# The paper's sensor graph: Table 3

**Graph notebook 1 in `notebooks/graph/`.** `notebooks/README.md` says how every notebook is laid out. `notebooks/graph/README.md` says what every graph notebook shares — which stored runs they read, which network and which window they follow — and defines the words used here. Notebook 02's step 9 introduces a graph layer from the beginning.

## What this notebook is for

The network can combine two sensors' readings in its graph layers only where the sensor graph joins them, so the graph settles what the network can and cannot see before training starts. This notebook shows the graph the paper designs by hand, exactly as a trained network received it: the table of which sensors are joined, the rescaled table the network actually multiplies by, what the first graph layer of that network does to one real window, how far three graph layers can carry information, and the stored five-seed result of the network trained on it.

## What it does not do

- **It does not train anything.** It loads the network seed 42 trained in a stored run, and reads all five seeds' results from that run.
- **It does not say whether this graph is better than another.** `04_comparison_fd001.ipynb` sets it beside the other two, under one configuration and with paired tests.
- **It does not check the graph against the engine.** The table is copied from the paper. The five rules the paper says it followed are listed in `src/thesis_gnn_rul_xai/graph/README.md`, and the one place where the paper's own text asks for an edge its table leaves out is written up in `docs/findings/F-07.md` (finding F-07).

## The notebook in one list

| # | Step | What goes in | What comes out |
|---|------|--------------|----------------|
| 1 | **Load** the stored network | the stored five-seed run of this graph | seed 42's network and its data, checked against the run |
| 2 | **Read** the table | the 14 x 14 table of zeros and ones | 41 edges and each sensor's neighbours, checked against the paper's Table 3 |
| 3 | **Rescale** it | the same table | the table the network multiplies by, and the sensors it cannot tell apart |
| 4 | **Run one graph layer by hand** | one real window and the trained layer's 32 learned numbers | 14 x 32 numbers, the same as the network's own layer gives |
| 5 | **Measure how far three layers reach** | the table and the trained network | the distance between every pair of sensors, the four pairs out of reach, and a test on one of them |
| 6 | **Read the stored result** | the five seeds' scores | the mean and spread on the test engines and on the held-back engines |

## 1. Background

### The graph, in the terms of this problem

- **A node is a sensor.** There are 14, in the order of the paper's Table 3: the temperatures at the outlets of the low-pressure compressor, the high-pressure compressor and the low-pressure turbine (`T24`, `T30`, `T50`); the pressure in the bypass duct (`P15`); the total and static pressures at the high-pressure compressor outlet (`P30`, `Ps30`); the fan and core speeds (`Nf`, `Nc`) and the corrected fan speed (`NRf`); the fuel flow divided by `Ps30` (`phi`); the bypass ratio (`BPR`); the bleed enthalpy (`htBleed`); and the coolant bleeds of the two turbines (`W31`, `W32`). The paper's Table 2 describes each.
- **An edge is a judgement.** The paper joins two sensors when, by its reading of how the engine works, the two measurements are physically related (its Section 2.3.1). `src/thesis_gnn_rul_xai/graph/README.md` lists the five rules it states.
- **What a node carries in.** In every run read here, each sensor carries one number into the graph layers: its last reading of the 30-cycle window, smoothed and rescaled (assumption A-07). So the first graph layer receives a table of shape `(14, 1)`: 14 rows, one per sensor, one number in each.

### One graph layer, on three sensors

The numbers in this example are made up, small enough to check by hand. Three sensors `a`, `b` and `c`; `a` is joined to `b`, and `b` to `c`.

```
A     = [[0, 1, 0],        A + I = [[1, 1, 0],        row totals: 2, 3, 2
         [1, 0, 1],                 [1, 1, 1],
         [0, 1, 0]]                 [0, 1, 1]]

Â     = [[1/2, 1/2,   0],           each row of A + I divided by its total
         [1/3, 1/3, 1/3],
         [  0, 1/2, 1/2]]
```

`A + I` adds 1 down the diagonal, so each sensor is also joined to itself and keeps its own reading in the mix. Dividing each row by its total makes every row add up to 1, so multiplying by `Â` takes an average. With one number per sensor, `H = [4, 1, -2]` (shape `(3, 1)`):

```
Â H   = [ (4 + 1) / 2,  (4 + 1 - 2) / 3,  (1 - 2) / 2 ]  =  [2.5, 1.0, -0.5]
```

That is the **mixing**: each sensor's number is replaced by the average over itself and its neighbours. Then the **transforming**: a learned table `W`, here of shape `(1, 2)`, turns each sensor's one number into two, and the same `W` serves every sensor:

```
W     = [[2, -1]]

Â H W = [[ 5.0, -2.5],        ReLU(Â H W) = [[5.0, 0.0],
         [ 2.0, -1.0],                       [2.0, 0.0],
         [-1.0,  0.5]]                       [0.0, 0.5]]
```

ReLU replaces every negative number with zero. The shape went from `(3, 1)` to `(3, 2)`. Sensors `a` and `c` are not joined, yet after this layer both hold part of `b`'s number; after a second layer `a` would hold part of what `c` held, passed on through `b`. That is why one layer reaches one edge, two layers reach two, and three reach three.

### The same thing in this network

The paper's Equation 8 writes the first graph layer as `H(1) = ReLU(Ã X W(1))`, its `Ã` being the rescaled table. These notebooks write that table `Â`, as notebook 02 does:

- `X` is what the sensors carry in, shape `(14, 1)`: each sensor's last reading.
- `Â` is the rescaled table, shape `(14, 14)`. It is fixed before training. The paper says only that it is normalised and includes each sensor's link to itself; dividing each row by its total is this project's choice (assumption A-21).
- `W(1)` is the layer's learned table, shape `(1, 32)`: 32 numbers in all, the same for every sensor.
- `H(1)` is what leaves, shape `(14, 32)`, and becomes what the second layer reads. The second layer's table is `(32, 64)` and the third's `(64, 32)`, so the graph layers end at `(14, 32)`, and the dense layers after them read those 448 numbers all at once.

**Do not confuse** the rescaled table `Â` with the learned table `W`. `Â` says *whose* numbers are averaged with whose, and training never changes it. `W` says *what is made* of each average, and training is what sets it. Neither is the readings themselves, which are what was measured, nor `H(1)`, which is what the network computed from all three.

## 2. Setup

The imports, then the stored runs, chosen by the rule `notebooks/graph/README.md` gives: the newest commit at which all three graphs have a stored five-seed run of `config/fd001.yaml` that saved its trained networks, and the newest run of each graph at that commit. Every notebook in this folder chooses the same way, so all four read the same runs. If a run is missing, the cell stops and prints the commands that make them.

In [1]:
from __future__ import annotations

import logging
import sys
from dataclasses import replace
from pathlib import Path

from IPython.display import HTML, display
import networkx as nx
import numpy as np
import pandas as pd

# The project's top folder is the nearest one above this notebook that holds
# pyproject.toml. Adding its src/ folder to Python's search path makes the
# package importable from a clone that was never installed.
PROJECT_ROOT = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "pyproject.toml").is_file()
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

try:
    import torch
except ImportError as missing:
    raise ImportError(
        "This notebook loads a trained network, which needs PyTorch, and PyTorch is "
        'not installed. Install the project with  pip install -e ".[model]"'
    ) from missing

from thesis_gnn_rul_xai.config import load_config
from thesis_gnn_rul_xai.docsite.figures import render_figure
from thesis_gnn_rul_xai.graph.adjacency import (
    PHYSICS_ADJACENCY_ROWS,
    adjacency_report,
    adjacency_to_frame,
    normalize_adjacency,
    pairs_beyond_reach,
    to_networkx,
)
from thesis_gnn_rul_xai.models import ModelConfig
from thesis_gnn_rul_xai.training import TrainingConfig, evaluate_model, find_runs, load_seed
from thesis_gnn_rul_xai.utils.logging_setup import configure_logging

configure_logging(logging.ERROR)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch     : {torch.__version__}")

Project root: D:\Main\University\Thesis\Code\Thesis GNN RUL XAI
PyTorch     : 2.13.0+cpu


In [2]:
config = load_config(PROJECT_ROOT / "config" / "fd001.yaml")
GRAPHS = ("physics", "complete", "correlation")
GRAPH = "physics"    # the graph this notebook shows
SEED = 42              # the network every worked example uses
COMMAND = "python scripts/train.py --config config/fd001.yaml --graph {} --seeds 5"
# The training settings a run records in its summary.json, at their default values.
TRAINING = {key: getattr(TrainingConfig(), key) for key in (
    "epochs", "batch_size", "learning_rate", "final_learning_rate", "early_stopping_patience")}


def stored_runs(kind: str) -> list:
    # Five seeds of config/fd001.yaml with only the graph changed, the trained
    # networks saved, the network sizes of the paper's Table 4, and the default
    # training settings.
    return [
        stored for stored in find_runs(
            preprocessing=replace(config, graph=kind), seeds=5, with_weights=True,
            split="varies_with_seed", early_stopping=True, lr_schedule="step",
            temporal_stage="last_cycle",
        )
        if stored.model_config == ModelConfig()
        and {key: stored.summary.get("training", {}).get(key) for key in TRAINING} == TRAINING
    ]


found = {kind: stored_runs(kind) for kind in GRAPHS}
shared = set.intersection(*({stored.summary["commit"] for stored in runs} for runs in found.values()))
if not shared:
    missing = [kind for kind, runs in found.items() if not runs]
    raise FileNotFoundError(
        ("reports/ holds no stored five-seed run, with its networks saved, for: "
         + ", ".join(missing) if missing
         else "The stored runs of the three graphs come from different commits.")
        + "\nFrom one committed state of the code, run\n\n"
        + "\n".join("    " + COMMAND.format(kind) for kind in GRAPHS)
        + "\n\nand run this notebook again."
    )

# The newest commit at which all three graphs were stored, and the newest run
# of each graph at that commit.
newest = {
    commit: max(stored.summary["finished_utc"] for runs in found.values()
                for stored in runs if stored.summary["commit"] == commit)
    for commit in shared
}
commit = max(shared, key=newest.get)
runs = {kind: [s for s in found[kind] if s.summary["commit"] == commit][-1] for kind in GRAPHS}
run = runs[GRAPH]

print(f"Commit of all three graphs' runs: {commit}")
for kind, stored in runs.items():
    print(f"  {kind:12s} {stored.name}")

Commit of all three graphs' runs: f386291-dirty
  physics      fd001_physics_row_cap_5seed_varsplit_cfgba82e3_f386291-dirty_20260914T150004Z
  complete     fd001_complete_row_cap_5seed_varsplit_cfgf21547_f386291-dirty_20260914T150527Z
  correlation  fd001_correlation_row_cap_5seed_varsplit_cfg7fdfce_f386291-dirty_20260914T150213Z


## 3. The trained network

### Step 1 — Load the stored network

**What.** Load the network that seed 42 trained in the run chosen above, from the `model.pt` saved beside it, and rebuild its data with the same seed.

**Why.** The worked examples below should show the graph and the layer a network of this project's reported runs actually used. A network trained here instead would be one seed and would record no commit.

**Check.** The loaded network must score the test engines exactly as the run recorded for seed 42, and the graph inside it must be the rescaled table the rebuilt data holds.

In [3]:
model, prepared = load_seed(run, SEED)
sensors = list(prepared.sensor_symbols)

recorded = float(run.results.set_index("seed").loc[SEED, "rmse"])
scored = evaluate_model(model, prepared.test)["rmse"]
same_network = bool(np.isclose(scored, recorded, rtol=0, atol=1e-6))
same_graph = bool(np.allclose(model.adjacency.numpy(), prepared.adjacency_normalized))
assert same_network and same_graph
print(f"OK  test RMSE {scored:.6f} cycles, as the run recorded for seed {SEED}.")
print("OK  The network's graph is the rescaled table of the rebuilt data.")

OK  test RMSE 16.694915 cycles, as the run recorded for seed 42.
OK  The network's graph is the rescaled table of the rebuilt data.


## 4. The graph, step by step

### Step 2 — Read the table

**What.** Print the table of zeros and ones the network's data was built with, and list each sensor's neighbours: the sensors its row marks with a 1. How many there are is the sensor's **degree**.

**Why.** Everything after this depends on it. A wrong entry would change every result while the code still ran without complaint. The paper prints the table twice, as the numbers of Table 3 and as the coloured grid of Fig. 3, and both were checked against the copy in `src/thesis_gnn_rul_xai/graph/adjacency.py`, cell by cell, when it was written; `src/thesis_gnn_rul_xai/graph/README.md` records that check.

**Check.** The table must be the copy of Table 3, equal its own mirror image across the diagonal (if `T24` is joined to `T30`, `T30` is joined to `T24`), hold zeros down the diagonal, and have 41 edges.

In [4]:
raw = prepared.adjacency_raw          # NumPy array, shape (14, 14), zeros and ones
display(adjacency_to_frame(raw.astype(int), sensors))

table_3 = np.array([PHYSICS_ADJACENCY_ROWS[sensor] for sensor in sensors])
report = adjacency_report(raw, sensors)
is_table_3 = bool((raw == table_3).all())
assert is_table_3 and report["is_symmetric"] and not report["has_self_loops"]
assert report["n_edges"] == 41
print(f"OK  The table is the copy of Table 3: {report['n_edges']} edges, its own mirror image, "
      "zeros down the diagonal.\n")

neighbours = pd.DataFrame({
    "degree": raw.sum(axis=1).astype(int),
    "joined to": [", ".join(sensors[j] for j in np.flatnonzero(row)) for row in raw],
}, index=sensors)
display(neighbours.sort_values("degree", ascending=False))

,T24,T30,T50,P15,P30,Nf,Nc,Ps30,phi,NRf,BPR,htBleed,W31,W32
T24,0,1,0,1,1,1,0,1,0,1,1,0,0,0
T30,1,0,1,0,1,0,1,1,1,0,0,1,1,0
T50,0,1,0,0,0,0,1,0,0,0,0,0,1,1
P15,1,0,0,0,0,1,0,0,0,1,1,0,0,0
P30,1,1,0,0,0,0,1,1,1,0,0,1,1,0
Nf,1,0,0,1,0,0,0,0,0,1,1,0,0,0
Nc,0,1,1,0,1,0,0,1,1,0,0,1,1,1
Ps30,1,1,0,0,1,0,1,0,1,0,0,1,1,0
phi,0,1,0,0,1,0,1,1,0,0,0,1,1,0
NRf,1,0,0,1,0,1,0,0,0,0,1,0,0,0


OK  The table is the copy of Table 3: 41 edges, its own mirror image, zeros down the diagonal.



,degree,joined to
T30,8,"T24, T50, P30, Nc, Ps30, phi, htBleed, W31"
Nc,8,"T30, T50, P30, Ps30, phi, htBleed, W31, W32"
W31,8,"T30, T50, P30, Nc, Ps30, phi, htBleed, W32"
htBleed,7,"T30, P30, Nc, Ps30, phi, W31, W32"
P30,7,"T24, T30, Nc, Ps30, phi, htBleed, W31"
T24,7,"T30, P15, P30, Nf, Ps30, NRf, BPR"
Ps30,7,"T24, T30, P30, Nc, phi, htBleed, W31"
phi,6,"T30, P30, Nc, Ps30, htBleed, W31"
T50,4,"T30, Nc, W31, W32"
P15,4,"T24, Nf, NRf, BPR"


Every sensor is joined to at least four others and at most eight. Each edge appears twice in the table, once in each sensor's row, so the 82 ones in it make 41 edges.

### Step 3 — Rescale it

**What.** Add 1 down the diagonal, joining each sensor to itself, then divide every row by its own total. `T24` has 7 neighbours, so with itself its row holds 8 ones, and each becomes 1/8 = 0.125.

**Why.** The network multiplies by this table, not by the zeros and ones. Without the 1 on the diagonal a sensor's own reading would be left out of its own average. Without the division, a sensor with 8 neighbours would receive a sum of 9 numbers where a sensor with 4 receives a sum of 5, and those sums would grow with every layer; divided by its total, every row takes an average.

**Check.** Every row must add up to 1, and the table must be what `normalize_adjacency(raw, method="row")` gives.

In [5]:
a_hat = prepared.adjacency_normalized          # NumPy array, shape (14, 14)
display(adjacency_to_frame(a_hat, sensors).round(3))

rows_sum_to_one = bool(np.allclose(a_hat.sum(axis=1), 1.0, atol=1e-6))
matches_function = bool(np.allclose(a_hat, normalize_adjacency(raw, method="row")))
assert rows_sum_to_one and matches_function
t24 = sensors.index("T24")
print(f"T24's row: {int(raw[t24].sum())} neighbours plus itself, so {int(raw[t24].sum()) + 1} entries "
      f"of {a_hat[t24, t24]:.4f} = 1/{int(raw[t24].sum()) + 1}.")
print("OK  Every row adds up to 1, and the table is what normalize_adjacency gives.")

,T24,T30,T50,P15,P30,Nf,Nc,Ps30,phi,NRf,BPR,htBleed,W31,W32
T24,0.125,0.125,0.000,0.125,0.125,0.125,0.000,0.125,0.000,0.125,0.125,0.000,0.000,0.000
T30,0.111,0.111,0.111,0.000,0.111,0.000,0.111,0.111,0.111,0.000,0.000,0.111,0.111,0.000
T50,0.000,0.200,0.200,0.000,0.000,0.000,0.200,0.000,0.000,0.000,0.000,0.000,0.200,0.200
P15,0.200,0.000,0.000,0.200,0.000,0.200,0.000,0.000,0.000,0.200,0.200,0.000,0.000,0.000
P30,0.125,0.125,0.000,0.000,0.125,0.000,0.125,0.125,0.125,0.000,0.000,0.125,0.125,0.000
Nf,0.200,0.000,0.000,0.200,0.000,0.200,0.000,0.000,0.000,0.200,0.200,0.000,0.000,0.000
Nc,0.000,0.111,0.111,0.000,0.111,0.000,0.111,0.111,0.111,0.000,0.000,0.111,0.111,0.111
Ps30,0.125,0.125,0.000,0.000,0.125,0.000,0.125,0.125,0.125,0.000,0.000,0.125,0.125,0.000
phi,0.000,0.143,0.000,0.000,0.143,0.000,0.143,0.143,0.143,0.000,0.000,0.143,0.143,0.000
NRf,0.200,0.000,0.000,0.200,0.000,0.200,0.000,0.000,0.000,0.200,0.200,0.000,0.000,0.000


T24's row: 7 neighbours plus itself, so 8 entries of 0.1250 = 1/8.
OK  Every row adds up to 1, and the table is what normalize_adjacency gives.


The same two tables as a picture, drawn by the same file as every diagram on the documentation site. The darker a square, the larger the number in it; hovering over a square shows its value.

In [6]:
# Drawn from the paper's Table 3. Handed over as HTML, not as an SVG image,
# because the documentation site reads notebooks with nbconvert, which keeps
# an HTML output and drops an SVG one.
display(HTML(render_figure("adjacency")))

#### Sensors the network cannot tell apart

Two rows of `Â` can be identical. That happens when two sensors are joined to exactly the same sensors, each other and themselves included: both then take the same average of the same numbers. From the first graph layer on, the two hold the same numbers, after every later graph layer too, because each layer averages identical rows into identical rows. The network can then use only what the group holds together, never one member on its own (finding F-18 measures this on the trained networks). The next cell finds every such group in Table 3.

In [7]:
def identical_rows(matrix: np.ndarray, names: list[str]) -> list[list[str]]:
    groups: dict[bytes, list[str]] = {}
    for name, row in zip(names, np.round(np.asarray(matrix, dtype=float), 9)):
        groups.setdefault(row.tobytes(), []).append(name)
    return [group for group in groups.values() if len(group) > 1]


groups = identical_rows(a_hat, sensors)
for group in groups:
    first = sensors.index(group[0])
    shared_set = [sensors[j] for j in np.flatnonzero(a_hat[first])]
    print(f"{', '.join(group):24s} each averages over the same {len(shared_set)}: {', '.join(shared_set)}")
n_different_rows = len(np.unique(np.round(a_hat.astype(float), 9), axis=0))
print(f"\n{n_different_rows} different rows among the 14.")

P15, Nf, NRf, BPR        each averages over the same 5: T24, P15, Nf, NRf, BPR
P30, Ps30                each averages over the same 8: T24, T30, P30, Nc, Ps30, phi, htBleed, W31
Nc, W31                  each averages over the same 9: T30, T50, P30, Nc, Ps30, phi, htBleed, W31, W32

9 different rows among the 14.


Three groups, holding eight sensors between them, so the 14 sensors leave the first graph layer as only 9 different rows of numbers. This is a property of which sensors Table 3 joins, not of the readings, and it holds whichever readings come in.

### Step 4 — One graph layer, by hand

**What.** Follow one real window through the first graph layer of the trained network. The window is the first test engine's: 14 sensors by 30 cycles, the last cycle being the engine's last recorded one. The temporal stage keeps each sensor's last reading, giving `X`, shape `(14, 1)`. Then `Â X` mixes, `W(1)` transforms and ReLU clips, worked out here with plain NumPy and compared with the network's own layer. Notebook 02's step 9 averaged `P15`'s row of this same window by hand; here `T24` is followed through the whole layer, the transforming and ReLU included.

**Why.** "The graph layer mixes each sensor with its neighbours" is a description. These are the numbers it describes.

**Check.** The hand computation must give the same 14 x 32 numbers as the network's first graph layer, to within rounding.

In [8]:
window = prepared.test.features[0]                 # NumPy array, shape (14, 30)
print(f"Test engine {prepared.test.units[0]}, window ending at cycle {prepared.test.end_cycles[0]}, "
      f"shape {window.shape}")

x = window[:, -1]                                  # shape (14,): what the temporal stage keeps
print("\nX, each sensor's last reading (smoothed and rescaled):")
print(pd.Series(x, index=sensors).round(4).to_frame("X").T.to_string())

mixed = a_hat @ x                                  # shape (14,): the mixing, Â X
joined = np.flatnonzero(a_hat[t24])
print(f"\nT24 averages {len(joined)} readings:")
print("   (" + " + ".join(f"{x[j]:.4f}" if x[j] >= 0 else f"({x[j]:.4f})" for j in joined)
      + f") / {len(joined)}")
print(f"   = {x[joined].sum():+.4f} / {len(joined)} = {x[joined].mean():+.4f}"
      f"     and Â X gives T24 {mixed[t24]:+.4f}")

w1 = model.graph_layers[0].weight.detach().numpy()   # shape (1, 32): learned in training
by_hand = np.maximum(mixed[:, None] * w1, 0.0)        # shape (14, 32): ReLU(Â X W(1))
print(f"\nW(1) has shape {w1.shape}. T24's average times its first six numbers, then ReLU:")
print(pd.DataFrame({
    "W(1)": w1[0, :6],
    "T24's average x W(1)": mixed[t24] * w1[0, :6],
    "after ReLU": by_hand[t24, :6],
}).round(4).T.to_string())
print(f"ReLU sets {int((by_hand[t24] == 0).sum())} of T24's 32 numbers to zero.")

with torch.no_grad():
    carried = model.temporal(torch.from_numpy(window[None]))           # tensor (1, 14, 1)
    by_layer = torch.relu(model.graph_layers[0](carried, model.adjacency))[0].numpy()
temporal_keeps_last = bool(np.allclose(carried[0, :, 0].numpy(), x))
layer_matches = bool(np.abs(by_hand - by_layer).max() < 1e-6)
assert temporal_keeps_last and layer_matches
print(f"\nOK  The temporal stage hands on the last reading, and the hand computation gives the "
      f"layer's {by_layer.shape[0]} x {by_layer.shape[1]} numbers "
      f"(largest difference {np.abs(by_hand - by_layer).max():.1e}).")
different_after_layer_1 = len(np.unique(np.round(by_layer.astype(float), 9), axis=0))
print(f"    {different_after_layer_1} different rows leave the layer, one per different row of Â.")

Test engine 1, window ending at cycle 31, shape (14, 30)

X, each sensor's last reading (smoothed and rescaled):
      T24     T30     T50    P15     P30      Nf     Nc    Ps30     phi     NRf     BPR  htBleed     W31     W32
X -0.4272 -1.2246 -1.1072  0.228  1.0624 -0.2107 -0.609 -0.9797  0.6584 -0.3687 -0.9152  -0.7095  0.5842  0.8267

T24 averages 8 readings:
   ((-0.4272) + (-1.2246) + 0.2280 + 1.0624 + (-0.2107) + (-0.9797) + (-0.3687) + (-0.9152)) / 8
   = -2.8358 / 8 = -0.3545     and Â X gives T24 -0.3545

W(1) has shape (1, 32). T24's average times its first six numbers, then ReLU:
                           0       1       2       3       4       5
W(1)                  0.3153  0.2952 -0.0098  0.3957 -0.0134  0.0020
T24's average x W(1) -0.1118 -0.1047  0.0035 -0.1403  0.0047 -0.0007
after ReLU            0.0000  0.0000  0.0035  0.0000  0.0047  0.0000
ReLU sets 15 of T24's 32 numbers to zero.

OK  The temporal stage hands on the last reading, and the hand computation gives th

Two things in that output are worth stopping on.

- **What `T24` carries out is decided by one number.** With one reading per sensor coming in, mixing gives each sensor one average, and the layer's 32 outputs for that sensor are that average times the 32 learned numbers of `W(1)`, with the negative products set to zero. A sensor whose average is negative keeps exactly the outputs whose learned number is negative, and the other way round. That holds for every window, not only this one: after the first graph layer, every sensor's 32 numbers are the size of its average times one of two fixed rows of numbers, the row chosen by the average's sign (finding F-21).
- **The groups of step 3 come out as identical rows.** 14 sensors went in and 9 different rows came out.

### Step 5 — How far three layers reach

**What.** Count, for every pair of sensors, the fewest edges between them. Then test the count with the trained network: raise `P15`'s readings by 1 — one standard deviation of `P15` over the training engines, the unit the readings were rescaled to — run the window through all three graph layers, and see which sensors' numbers change.

**Why.** One layer moves information one edge, so with three layers a pair of sensors four edges apart cannot affect each other anywhere in the graph layers, whatever training does. That is a limit set by the table before any training.

**Check.** `W32`, four edges from `P15`, must not change at all — exactly zero, not merely small — and every sensor within three edges of `P15` must change.

In [9]:
# Drawn from the paper's Table 3; until 2026-10-02 it was in notebook 02's section 5.
display(HTML(render_figure("reach")))

print(f"Fewest edges between two sensors, and how many of the 91 pairs: {report['distance_histogram']}")
print(f"Largest distance: {report['diameter']} edges")
beyond = pairs_beyond_reach(raw, sensors, n_layers=len(model.graph_layers))
print(f"Pairs further apart than {len(model.graph_layers)} edges: "
      + ", ".join(f"{a}-{b}" for a, b, _ in beyond))
print(f"Sensors whose removal would split the graph in two: {report['articulation_points']}")
without_t24 = to_networkx(raw, sensors)
without_t24.remove_node("T24")
print("Without T24 the graph falls into: "
      + " | ".join(", ".join(sorted(piece)) for piece in nx.connected_components(without_t24)))

Fewest edges between two sensors, and how many of the 91 pairs: {1: 41, 2: 25, 3: 21, 4: 4}
Largest distance: 4 edges
Pairs further apart than 3 edges: BPR-W32, NRf-W32, Nf-W32, P15-W32
Sensors whose removal would split the graph in two: ['T24']
Without T24 the graph falls into: Nc, P30, Ps30, T30, T50, W31, W32, htBleed, phi | BPR, NRf, Nf, P15


All four pairs out of reach have `W32` at one end, and at the other end a member of the first group of step 3 — `P15`, `Nf`, `NRf` and `BPR`. `T24` is the only sensor whose removal would split the graph in two: on one side the fan sensors, on the other the core sensors, so everything that passes between the two groups in the graph layers passes through `T24`.

In [10]:
def three_graph_layers(windows: torch.Tensor) -> torch.Tensor:
    # Through the temporal stage first, exactly as the network's forward pass does.
    hidden = model.temporal(windows)
    for layer in model.graph_layers:
        hidden = torch.relu(layer(hidden, model.adjacency))
    return hidden


base = torch.from_numpy(window[None].copy())
bumped = base.clone()
bumped[0, sensors.index("P15")] += 1.0
with torch.no_grad():
    before, after = three_graph_layers(base)[0], three_graph_layers(bumped)[0]
    prediction_before = float(model.predict_cycles(base)[0])
    prediction_after = float(model.predict_cycles(bumped)[0])

change = (after - before).abs().amax(dim=1).numpy()   # largest change among each sensor's 32 numbers
distance = nx.single_source_shortest_path_length(to_networkx(raw, sensors), "P15")
reach_table = pd.DataFrame({
    "edges from P15": [distance[sensor] for sensor in sensors],
    "largest change after three graph layers": change,
}, index=sensors).sort_values(["edges from P15", "largest change after three graph layers"],
                              ascending=[True, False])
display(reach_table.round(6))

unreachable_unchanged = bool(change[sensors.index("W32")] == 0.0)
reachable_changed = all(change[sensors.index(s)] > 0 for s in sensors if distance[s] <= 3)
assert unreachable_unchanged and reachable_changed
print("OK  W32, four edges away, is unchanged to the last digit; every sensor within three edges moved.")
print(f"\nThe prediction itself: {prediction_before:.3f} cycles before, {prediction_after:.3f} after.")

,edges from P15,largest change after three graph layers
P15,0,0.060703
Nf,1,0.060703
NRf,1,0.060703
BPR,1,0.060703
T24,1,0.039941
P30,2,0.007128
Ps30,2,0.007128
T30,2,0.006336
phi,3,0.002287
htBleed,3,0.002002


OK  W32, four edges away, is unchanged to the last digit; every sensor within three edges moved.

The prediction itself: 108.619 cycles before, 107.884 after.


The changes fall away with distance: `P15` and its four neighbours move the most, sensors two and three edges away less, and `W32` not at all.

**Do not confuse** "out of reach in the graph layers" with "no effect on the prediction". After the graph layers, the dense layers read all 14 sensors' numbers at once, so a change to `P15` can still move the prediction, and the last line above shows it did. What the table rules out is narrower: `W32`'s own 32 numbers can never carry anything of `P15`'s, so the network can never form a combination of the two inside the graph layers.

## 5. The stored result

### Step 6 — Read the stored result

**What.** The scores of all five seeds of the run, on the four measures notebook 02's step 11 defines, and the error on each seed's held-back engines.

**Why.** One seed is not a result (working rule 3). The mean over five seeds with its standard deviation is what this project reports for this configuration.

**Check.** The means and spreads computed here must agree with the ones the run wrote into its own `summary.json`.

In [11]:
results = run.results.set_index("seed")
columns = {
    "rmse": "test RMSE", "score": "test NASA score", "r2": "test R-squared",
    "asue": "test ASUE", "validation_rmse": "held-back RMSE", "best_epoch": "pass kept",
}
per_seed = results[list(columns)].rename(columns=columns)
measures = list(columns.values())[:5]
shown = per_seed.astype(object)
for m in measures:
    shown[m] = [f"{value:.3f}" for value in per_seed[m]]
shown.loc["mean ± standard deviation"] = (
    [f"{per_seed[m].mean():.3f} ± {per_seed[m].std(ddof=1):.3f}" for m in measures] + [""]
)
print(f"Run    : {run.name}")
print(f"Commit : {run.summary['commit']}, finished {run.summary['finished_utc']} (UTC)\n")
display(shown)

# The run wrote its own mean and spread into summary.json; they must agree.
matches_summary = all(
    np.isclose(results[m].mean(), run.summary["mean"][m])
    and np.isclose(results[m].std(ddof=1), run.summary["std"][m])
    for m in ("rmse", "score", "r2", "asue")
)
assert matches_summary
print("OK  The means and spreads agree with the run's own summary.json.")

Run    : fd001_physics_row_cap_5seed_varsplit_cfgba82e3_f386291-dirty_20260914T150004Z
Commit : f386291-dirty, finished 20260914T150004Z (UTC)



,test RMSE,test NASA score,test R-squared,test ASUE,held-back RMSE,pass kept
seed,,,,,,
42,16.695,580.398,0.826,12.253,20.821,51
43,16.131,565.285,0.838,10.314,19.831,55
44,16.843,550.748,0.823,12.716,21.529,8
45,15.970,465.051,0.841,10.929,22.066,18
46,16.843,537.625,0.823,12.044,21.594,35
mean ± standard deviation,16.496 ± 0.415,539.822 ± 44.748,0.830 ± 0.008,11.651 ± 0.995,21.168 ± 0.870,


OK  The means and spreads agree with the run's own summary.json.


**Which published figure this may be set beside.** These runs give each sensor its last reading and have no recurrent stage, so they compare with the paper's "ESTGNN w/o LSTM" row, 14.93 on FD001 in its Table 11, and never with the 11.78 of its full model. Two more differences go with any such comparison, neither of which the paper describes. The test labels here are capped at 125 cycles, which can only lower the RMSE and the NASA score set beside the paper's, and gives no such guarantee for R-squared or ASUE (assumption A-02). And the parameters kept are those of the pass over the training data that did best on the held-back engines (assumptions A-06 and A-18).

(Corrected 2026-10-02, before this notebook was first committed: this paragraph said capping could only move a number here in the network's favour, the overstatement that assumption A-02 withdrew the same day for two of the four measures. A review the same day found it.)

**The held-back error is not comparable with the test error.** It is computed over every window of 20 engines rather than one window per engine, and it chose which pass's parameters were kept, so it is no fair score of the network (assumption A-06). `04_comparison_fd001.ipynb` uses it for what it is good for: comparing two graphs on engines the test score was never consulted about.

## 6. Final verification

Every check in this notebook, in one place.

In [12]:
checks = [
    ("The loaded network scores what the run recorded", same_network, f"seed {SEED}"),
    ("The network's graph is the rebuilt data's", same_graph, run.summary["commit"]),
    ("The table is the copy of Table 3", is_table_3, f"{report['n_edges']} edges"),
    ("Every row of the rescaled table adds up to 1", rows_sum_to_one, "row rescaling"),
    ("The rescaled table is normalize_adjacency's", matches_function, 'method="row"'),
    ("The temporal stage hands on the last reading", temporal_keeps_last, "test engine 1"),
    ("One graph layer by hand equals the network's", layer_matches,
     f"{different_after_layer_1} different rows out"),
    ("W32, four edges from P15, does not move", unreachable_unchanged, "three graph layers"),
    ("Every sensor within three edges of P15 moves", reachable_changed, "three graph layers"),
    ("Five-seed means agree with summary.json", matches_summary, run.name[:40] + "..."),
]

width = max(len(name) for name, _, _ in checks) + 2
print(f"{'CHECK':<{width}} RESULT   DETAIL")
print("-" * (width + 40))
for name, passed, detail in checks:
    print(f"{name:<{width}} {'PASS' if passed else 'FAIL':<8} {detail}")
print("-" * (width + 40))
failed = [name for name, passed, _ in checks if not passed]
if failed:
    raise AssertionError(f"{len(failed)} check(s) failed: {failed}")
print(f"All {len(checks)} checks passed.")

CHECK                                             RESULT   DETAIL
-----------------------------------------------------------------------------------------
The loaded network scores what the run recorded   PASS     seed 42
The network's graph is the rebuilt data's         PASS     f386291-dirty
The table is the copy of Table 3                  PASS     41 edges
Every row of the rescaled table adds up to 1      PASS     row rescaling
The rescaled table is normalize_adjacency's       PASS     method="row"
The temporal stage hands on the last reading      PASS     test engine 1
One graph layer by hand equals the network's      PASS     9 different rows out
W32, four edges from P15, does not move           PASS     three graph layers
Every sensor within three edges of P15 moves      PASS     three graph layers
Five-seed means agree with summary.json           PASS     fd001_physics_row_cap_5seed_varsplit_cfg...
-------------------------------------------------------------------------------

## 7. Summary

### What this notebook produced

- **The paper's graph, as the stored network received it**: 41 edges, every sensor joined to between four and eight others, and the rescaled table in which three groups of sensors have identical rows, so that 14 sensors leave every graph layer as 9 different rows of numbers.
- **One graph layer worked by hand** on the first test engine's window with the trained network's own learned numbers, and matched to the layer's output.
- **The reach of three layers**: four pairs of sensors, all with `W32` at one end, are out of reach, and a change to `P15` leaves `W32` unchanged to the last digit.
- **The stored five-seed result** of the network trained on this graph, with the commit that produced it.

### Decisions that were not read from the paper

| Decision | Value used | Written up as |
|---|---|---|
| How the table is rescaled | Each row of `A + I` divided by its own total | A-21 |
| What a sensor carries into the graph layers | The last reading of its window, one number | A-07 |
| Held-back engines and the parameters kept | 20 % of the training engines; the parameters of the pass that did best on them | A-06 |
| The edge between `phi` and `W32` | Left out, as the table prints it, though the paper's text asks for it | F-07 |
| Which stored runs this folder reads | The newest commit at which all three graphs were stored | `notebooks/graph/README.md` |

### What comes next

`02_complete_fd001.ipynb` and `03_correlation_fd001.ipynb` take the same steps on the two other graphs, and `04_comparison_fd001.ipynb` sets the three results side by side.

### References

- Wen, J.-R., et al. (2026). Explainable spatiotemporal graph neutral network with dynamic SHAP enhancement for aeroengine RUL prediction. *Aerospace Science and Technology*, 168, 110875.
- Kipf, T. N., & Welling, M. (2017). Semi-supervised classification with graph convolutional networks. *ICLR*.

Full citations are in `docs/REFERENCES.md`.